In [1]:
import pandas as pd
import numpy as np

### Reading in master hospital table:

In [2]:
hospitals_master = pd.read_csv('../data/hospitals_master.csv')

In [3]:
hospitals_master.shape

(2276, 18)

In [4]:
hospitals_master.columns

Index(['Facility Name', 'State', 'Zip', 'County', 'Emergency Services',
       'Hospital Ownership', 'Closed', 'Closure Date', 'Full Address',
       'Prior Name', 'RUCA', 'Tract_Code', 'CBSA_Code', 'CBSA_Title',
       'Metro_Status', 'County_FIPS', 'State_FIPS', 'CCN'],
      dtype='object')

In [5]:
hospitals_master['Closed'].value_counts()

Closed
0    2085
2     106
1      85
Name: count, dtype: int64

I'll need to filter out any hospitals that closed before 2010, since the financial data I'll be using only goes back to 2010. This will affect 5 hospitals (13% of the closed or converted hospitals in the dataset). Also, starting our data window at 2010 ensures that every hospital in the final sample has at least one year of data prior to the year of closure.

In [6]:
hospitals_master['Closure Date'] = pd.to_datetime(hospitals_master['Closure Date'])

In [7]:
hospitals_master[~hospitals_master['Closure Date'].isna()]['Closure Date'].sort_values()

986    2005-03-01
688    2005-03-01
1257   2005-06-01
507    2005-07-01
602    2005-07-01
          ...    
1020   2025-05-01
1358   2025-05-01
658    2025-09-01
428    2025-10-01
298    2026-05-01
Name: Closure Date, Length: 188, dtype: datetime64[ns]

In [8]:
hospitals_master = hospitals_master[~(hospitals_master['Closure Date'] < '2010-01-01')]

In [9]:
hospitals_master.isna().sum()

Facility Name            0
State                    0
Zip                      0
County                   0
Emergency Services     123
Hospital Ownership     124
Closed                   0
Closure Date          2088
Full Address             0
Prior Name            2235
RUCA                     0
Tract_Code               0
CBSA_Code               69
CBSA_Title              69
Metro_Status             0
County_FIPS              0
State_FIPS               0
CCN                      0
dtype: int64

In [10]:
hospitals_master.head()

,Facility Name,State,Zip,County,Emergency Services,Hospital Ownership,Closed,Closure Date,Full Address,Prior Name,RUCA,Tract_Code,CBSA_Code,CBSA_Title,Metro_Status,County_FIPS,State_FIPS,CCN
0,ABBEVILLE GENERAL HOSPITAL,LA,70510,VERMILION,Yes,Government - Hospital District or Authority,0,NaT,"118 N HOSPITAL DR, ABBEVILLE, LA, 70510",NaN,4.0,950901.0,29180.0,"Lafayette, LA",Metropolitan Statistical Area,22113,22.0,190034
1,ABBOTT NORTHWESTERN HOSPITAL,MN,55407,HENNEPIN,Yes,Voluntary non-profit - Private,0,NaT,"800 EAST 28TH STREET, MINNEAPOLIS, MN, 55407",NaN,1.0,125800.0,33460.0,"Minneapolis-St. Paul-Bloomington, MN-WI",Metropolitan Statistical Area,27053,27.0,240057
2,ABRAZO ARROWHEAD HOSPITAL,AZ,85308,MARICOPA,Yes,Proprietary,0,NaT,"18701 NORTH 67TH AVENUE, GLENDALE, AZ, 85308",NaN,1.0,615900.0,38060.0,"Phoenix-Mesa-Chandler, AZ",Metropolitan Statistical Area,4013,4.0,30094
3,ABRAZO CENTRAL CAMPUS,AZ,85015,MARICOPA,Yes,Physician,0,NaT,"2000 WEST BETHANY HOME ROAD, PHOENIX, AZ, 85015",NaN,1.0,106802.0,38060.0,"Phoenix-Mesa-Chandler, AZ",Metropolitan Statistical Area,4013,4.0,30030
4,ABRAZO SCOTTSDALE CAMPUS,AZ,85032,MARICOPA,Yes,Proprietary,0,NaT,"3929 EAST BELL ROAD, PHOENIX, AZ, 85032",NaN,1.0,103303.0,38060.0,"Phoenix-Mesa-Chandler, AZ",Metropolitan Statistical Area,4013,4.0,30083


### Gathering hospital financial data:

Hospital financial data aggregation was performed in the "Financial Report Data Gathering Notebook" notebook with the help of code drawn from the HCRIS-databuilder GitHub project (https://github.com/Rush-Quality-Analytics/HCRIS-databuilder/tree/master).

In [11]:
financials = pd.read_csv('../data/HCRIS_databuilder/filtered_datasets/HCRIS_filtered.csv').drop(columns=['Report_Period_Begin_Yr','Urban (1) and Rural (2) Indicator','State']).rename(columns={'Beginning FFY':'Year',': General Ownership Type':'General Ownership Type'})

                                                                                                         

In [12]:
financials.columns

Index(['Hospital', 'Hospital type', 'Control type',
       'Donations, Land Improvements', 'NUMBER OF BEDS: Total Hospital',
       'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)',
       'Total Bad Debt expense', 'Fiscal Year Begin Date',
       'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)',
       'BALANCE SHEET: Inventory (G_C1THRU4_7)',
       'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)',
       'Fiscal Year End Date', 'Net Revenue from Medicaid', 'Total Charges',
       'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)',
       'IPPS Interim payment', 'Medicaid charges', 'Total Costs',
       'Total Inpatient Days', 'ADJUSTED SALARIES, Subtotal Salaries',
       'REIMBURSEMENT SETTLEMENT: Interim payments',
       'Cost of Uncompensated Care',
       'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)',
       'BED DAYS: Total Hospital',
       'BALANCE SHEET: Total Assets (G_C1THRU4_36)', 'NUMBER OF BEDS: ICU',
       'NUMBER OF BEDS: Adult

In [13]:
# Separating Hospital column into Facility Name and CCN, then dropping Hospital column
financials[['Facility Name', 'CCN']] = financials['Hospital'].str.extract(r'(.*)\s+\((\d+)\)')
financials = financials.drop(columns='Hospital')

In [14]:
# Creating a report period column to use later
financials['Fiscal Year End Date'] = pd.to_datetime(financials['Fiscal Year End Date'])
financials['Fiscal Year Begin Date'] = pd.to_datetime(financials['Fiscal Year Begin Date'])
financials['Report Period (Months)'] = (financials['Fiscal Year End Date'].dt.to_period('M') - financials['Fiscal Year Begin Date'].dt.to_period('M')).apply(lambda x: x.n)


In [15]:
financials.head(2)

,Hospital type,Control type,"Donations, Land Improvements",NUMBER OF BEDS: Total Hospital,BALANCE SHEET: Total Current Assets (G_C1THRU4_11),Total Bad Debt expense,Fiscal Year Begin Date,"RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)",BALANCE SHEET: Inventory (G_C1THRU4_7),BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),...,Financial Indicators: SOLVENCY Debt-to-Equity Ratio,Financial Indicators: SOLVENCY Debt Ratio,Financial Indicators: SOLVENCY Equity Ratio,Financial Indicators: SOLVENCY Interest Coverage Ratio,Financial Indicators: SOLVENCY total assets less total liabilities,Financial Indicators: EFFICIENCY asset turnover ratio,Financial Indicators: EFFICIENCY Accounts Receivable Turnover Ratio,Facility Name,CCN,Report Period (Months)
0,General Short Term (Acute Care Hospitals),Governmental-Hospital District,NaN,44.0,12705192.0,5211785.0,2011-01-01,1618104.0,733401.0,935269.0,...,0.335083,0.250983,0.749017,-24.782592,19958160.0,1.047686,5.801942,ABBEVILLE GENERAL HOSPITAL,190034,11
1,General Short Term (Acute Care Hospitals),Governmental-Hospital District,NaN,44.0,14373750.0,5321114.0,2012-01-01,1712816.0,847101.0,85345.0,...,0.245467,0.197088,0.802912,-66.832730,23446756.0,0.878510,7.239000,ABBEVILLE GENERAL HOSPITAL,190034,11


Ensuring we have one row per hospital per year:

In [16]:
financials[financials[['Facility Name','Year']].duplicated(keep=False)][['Facility Name','Year','Fiscal Year Begin Date','Fiscal Year End Date','Total Bad Debt expense','Total cost of charity care']]


,Facility Name,Year,Fiscal Year Begin Date,Fiscal Year End Date,Total Bad Debt expense,Total cost of charity care
410,ADVENTIST LAGRANGE MEMORIAL HOSPITAL,2015,2015-01-01,2015-01-31,271409.0,227286.0
411,ADVENTIST LAGRANGE MEMORIAL HOSPITAL,2015,2015-02-01,2015-12-31,3783813.0,2116490.0
537,AHN HEMPFIELD NEIGHBORHOOD HOSPITAL,2022,2022-01-01,2022-06-30,NaN,14072.0
538,AHN HEMPFIELD NEIGHBORHOOD HOSPITAL,2022,2022-07-01,2023-06-30,NaN,21872.0
541,AHN WEXFORD HOSPITAL,2022,2021-12-10,2022-06-30,215610.0,18970.0
...,...,...,...,...,...,...
30228,YORK HOSPITAL,2020,2020-07-01,2021-06-30,46505049.0,4829032.0
30229,YORK HOSPITAL,2021,2021-07-01,2022-06-30,50623264.0,6060012.0
30230,YORK HOSPITAL,2022,2022-07-01,2023-06-30,43499866.0,7024589.0
30231,YORK HOSPITAL,2023,2023-07-01,2024-06-30,NaN,NaN


We see that some rows have duplicated hospital-years. This is either because "(1) each cost report covers part of the year
or (2) the second submission replaces the first submission". 

Because very few providers submit partial years, I will simplify the process, only keeping self-defined full year” cost reports. For our purposes, we define a “full year” as a report period having ten or more months. 

The other option would be to combine these reports with partial-year reporting periods, summing the appropriate fields, such as costs and revenue, to obtain a full year of data. We'd have to be cautious not to add certain numeric fields such as beds and FTEs since they are not meant to be accumulated.

Source for this information: Source: "HANDLING MULTIPLE REPORTS FOR A PROVIDER" here: https://www.lexjansen.com/sesug/2018/SESUG2018_Paper-287_Final_PDF.pdf

In [17]:
financials.shape

(30261, 65)

In [18]:
financials = financials[financials['Report Period (Months)'] >= 10]

In [19]:
financials.shape

(29385, 65)

In [20]:
(12116-11716)/12116

0.03301419610432486

We lose only about 3% of our dataset after filtering out short reporting periods.

Re-checking for duplicate hospital-years:

In [21]:
financials[financials[['Facility Name','Year']].duplicated(keep=False)][['Facility Name','Year','Fiscal Year Begin Date','Fiscal Year End Date','Report Period (Months)','Total Bad Debt expense','Total cost of charity care']].sort_values(by=['Facility Name','Year'])


,Facility Name,Year,Fiscal Year Begin Date,Fiscal Year End Date,Report Period (Months),Total Bad Debt expense,Total cost of charity care
996,ARIZONA GENERAL HOSPITAL,2019,2019-01-01,2019-12-31,11,38347641.0,7647738.0
1002,ARIZONA GENERAL HOSPITAL,2019,2019-01-15,2019-12-31,11,3983289.0,1551065.0
997,ARIZONA GENERAL HOSPITAL,2020,2020-01-01,2020-12-31,11,17062007.0,10332812.0
1003,ARIZONA GENERAL HOSPITAL,2020,2020-01-01,2020-12-31,11,6347120.0,3413819.0
998,ARIZONA GENERAL HOSPITAL,2021,2021-01-01,2021-12-31,11,12584303.0,4583551.0
...,...,...,...,...,...,...,...
30230,YORK HOSPITAL,2022,2022-07-01,2023-06-30,11,43499866.0,7024589.0
30216,YORK HOSPITAL,2023,2023-01-01,2023-12-31,11,NaN,NaN
30231,YORK HOSPITAL,2023,2023-07-01,2024-06-30,11,NaN,NaN
30217,YORK HOSPITAL,2024,2024-01-01,2024-12-31,11,NaN,NaN


These duplicates likely result from a hospital updating or correcting the data initially reported. When this happens, it is common to use the most recently submitted (and likely most up-to-date) report and drop the old report to avoid double counting. That is what I will do here.

In [22]:
financials = (
    financials.sort_values(by=['Fiscal Year End Date'])
    .drop_duplicates(subset=['Facility Name', 'CCN','Year'], keep='last')
)

Re-checking for duplicate hospital-years:

In [23]:
financials[financials[['Facility Name','Year']].duplicated(keep=False)][['Facility Name','Year','Fiscal Year Begin Date','Fiscal Year End Date','Report Period (Months)','Total Bad Debt expense','Total cost of charity care']].sort_values(by=['Facility Name','Year'])


,Facility Name,Year,Fiscal Year Begin Date,Fiscal Year End Date,Report Period (Months),Total Bad Debt expense,Total cost of charity care
1002,ARIZONA GENERAL HOSPITAL,2019,2019-01-15,2019-12-31,11,3983289.0,1551065.0
996,ARIZONA GENERAL HOSPITAL,2019,2019-01-01,2019-12-31,11,38347641.0,7647738.0
997,ARIZONA GENERAL HOSPITAL,2020,2020-01-01,2020-12-31,11,17062007.0,10332812.0
1003,ARIZONA GENERAL HOSPITAL,2020,2020-01-01,2020-12-31,11,6347120.0,3413819.0
1004,ARIZONA GENERAL HOSPITAL,2021,2021-01-01,2021-12-31,11,13156537.0,3759846.0
...,...,...,...,...,...,...,...
30230,YORK HOSPITAL,2022,2022-07-01,2023-06-30,11,43499866.0,7024589.0
30216,YORK HOSPITAL,2023,2023-01-01,2023-12-31,11,NaN,NaN
30231,YORK HOSPITAL,2023,2023-07-01,2024-06-30,11,NaN,NaN
30217,YORK HOSPITAL,2024,2024-01-01,2024-12-31,11,NaN,NaN


I've confirmed we have 1 row per hospital-year. Dropping the FY begin/end dates and the report period.

In [24]:
financials = financials.drop(columns=['Fiscal Year Begin Date','Fiscal Year End Date','Report Period (Months)'])

We want to make sure we still have rows for missing hospital-years, so that we can interpolate missing values later.

In [25]:
financials['Year'].min()

2010

In [26]:
financials['Year'].max()

2025

In [27]:
years = pd.Series(list(range(2010, 2026)), name='Year')
hospitals = hospitals_master[['Facility Name','CCN']]
hospital_years = pd.merge(years, hospitals, how="cross")
hospital_years.sort_values(by=['Facility Name','Year'])

,Year,Facility Name,CCN
0,2010,ABBEVILLE GENERAL HOSPITAL,190034
2235,2011,ABBEVILLE GENERAL HOSPITAL,190034
4470,2012,ABBEVILLE GENERAL HOSPITAL,190034
6705,2013,ABBEVILLE GENERAL HOSPITAL,190034
8940,2014,ABBEVILLE GENERAL HOSPITAL,190034
...,...,...,...
26819,2021,ZUCKERBERG SAN FRANCISCO GENERAL HOSP & TRAUMA...,50228
29054,2022,ZUCKERBERG SAN FRANCISCO GENERAL HOSP & TRAUMA...,50228
31289,2023,ZUCKERBERG SAN FRANCISCO GENERAL HOSP & TRAUMA...,50228
33524,2024,ZUCKERBERG SAN FRANCISCO GENERAL HOSP & TRAUMA...,50228


In [28]:
hospital_years['CCN'] = hospital_years['CCN'].astype(str)

In [29]:
hospital_years['CCN'] = hospital_years['CCN'].str.zfill(6)

In [30]:
financials_full = hospital_years.merge(financials, how='left', on=['Facility Name','CCN','Year'])

Merging master hospital list with financial data:

In [31]:
hospitals_master['CCN'] = hospitals_master['CCN'].astype(str)
hospitals_master['CCN'] = hospitals_master['CCN'].str.zfill(6)

In [32]:
hospitals_full = financials_full.drop(columns=['Facility Name']).merge(hospitals_master, how='right', on='CCN')

In [33]:
hospitals_full.sort_values(by=['Facility Name','Year'])[['Facility Name','CCN','Year','Financial Indicators: Net Profit Margin','Financial Indicators: SOLVENCY Debt Ratio']].head(10)


,Facility Name,CCN,Year,Financial Indicators: Net Profit Margin,Financial Indicators: SOLVENCY Debt Ratio
0,ABBEVILLE GENERAL HOSPITAL,190034,2010,NaN,NaN
1,ABBEVILLE GENERAL HOSPITAL,190034,2011,0.259837,0.250983
2,ABBEVILLE GENERAL HOSPITAL,190034,2012,0.171192,0.197088
3,ABBEVILLE GENERAL HOSPITAL,190034,2013,0.029877,0.208432
4,ABBEVILLE GENERAL HOSPITAL,190034,2014,-0.052003,0.266827
5,ABBEVILLE GENERAL HOSPITAL,190034,2015,0.130972,0.170813
6,ABBEVILLE GENERAL HOSPITAL,190034,2016,0.049917,0.264305
7,ABBEVILLE GENERAL HOSPITAL,190034,2017,0.143066,0.267149
8,ABBEVILLE GENERAL HOSPITAL,190034,2018,-0.003600,0.256604
9,ABBEVILLE GENERAL HOSPITAL,190034,2019,0.033959,0.227741


We need to make sure and drop any financial data reported after a hospital scales down operations.

In [34]:
condition = (hospitals_full['Year'] <= hospitals_full['Closure Date'].dt.year) | hospitals_full['Closure Date'].isna()
hospitals_full = hospitals_full[condition]

In [35]:
(hospitals_full['Closure Date'].dt.year - hospitals_full['Year']).value_counts()

0.0     146
1.0     144
2.0     140
3.0     131
4.0     118
5.0     104
6.0      88
7.0      78
8.0      70
9.0      57
10.0     41
11.0     25
12.0     23
13.0     18
14.0     11
15.0      6
16.0      1
Name: count, dtype: int64

In [36]:
# Creating status target for survival regression
hospitals_full['Status'] = (hospitals_full['Closure Date'].dt.year == hospitals_full['Year'])

In [37]:
hospitals_full.shape

(34609, 79)

In [38]:
len(hospitals_full['CCN'].unique())

2235

In [39]:
hospitals_full.columns

Index(['Year', 'CCN', 'Hospital type', 'Control type',
       'Donations, Land Improvements', 'NUMBER OF BEDS: Total Hospital',
       'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)',
       'Total Bad Debt expense',
       'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)',
       'BALANCE SHEET: Inventory (G_C1THRU4_7)',
       'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)',
       'Net Revenue from Medicaid', 'Total Charges',
       'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)',
       'IPPS Interim payment', 'Medicaid charges', 'Total Costs',
       'Total Inpatient Days', 'ADJUSTED SALARIES, Subtotal Salaries',
       'REIMBURSEMENT SETTLEMENT: Interim payments',
       'Cost of Uncompensated Care',
       'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)',
       'BED DAYS: Total Hospital',
       'BALANCE SHEET: Total Assets (G_C1THRU4_36)', 'NUMBER OF BEDS: ICU',
       'NUMBER OF BEDS: Adults & Pediatrics',
       'TRIAL BALANCE OF EXPEN

In [40]:
hospitals_full['Hospital Type'] = hospitals_full['Hospital Ownership'].combine_first(hospitals_full['Control type'])
hospitals_full = hospitals_full.drop(columns=['Hospital Ownership','Hospital type','Control type'])
                                              

In [41]:
hospitals_full['Hospital Type'].isna().sum()

np.int64(686)

### Gathering hospital quality data:

Hospital quality data includes the following:

Process of Care Scores

Shares of patients receiving evidence-based treatments for AMI, CHF, pneumonia, surgical care, and outpatient care. All-payer. 

HCAHPS

Average scores for the aggregated questions in HCAHPS patient satisfaction survey. All-payer. 

Mortality and Readmission 

Estimates of AMI, CHF, and pneumonia mortality and readmission rates. Medicare FFS patients only


Source: https://github.com/asacarny/hospital-compare/

In [42]:
# Exporting our ccns of interest in order to filter hospital quality data
pd.Series(hospitals_full['CCN'].unique()).to_csv('../data/ccns.txt',index=False)

POC data:

In [43]:
hosp_quality_poc = pd.read_csv('../hospital-compare-master/output/poc.csv').rename(columns={'pn':'CCN','year':'Year'})

In [44]:
hosp_quality_poc.columns

Index(['CCN', 'ami1_denom', 'ami2_denom', 'ami3_denom', 'ami4_denom',
       'ami5_denom', 'ami7_denom', 'ami8_denom', 'cac1_denom', 'cac2_denom',
       'cac3_denom', 'hf1_denom', 'hf2_denom', 'hf3_denom', 'hf4_denom',
       'op2_denom', 'op3_denom', 'op4_denom', 'op5_denom', 'op6_denom',
       'op7_denom', 'pn2_denom', 'pn3_denom', 'pn4_denom', 'pn5_denom',
       'pn6_denom', 'pn7_denom', 'scipcard2_denom', 'scipinf1_denom',
       'scipinf2_denom', 'scipinf3_denom', 'scipinf4_denom', 'scipinf6_denom',
       'scipinf9_denom', 'scipvte1_denom', 'scipvte2_denom', 'Year',
       'op2_share', 'op4_share', 'op6_share', 'ami10_denom', 'op1_denom',
       'scipinf10_denom', 'ami10_share', 'ami2_share', 'ami7_share',
       'ami8_share', 'cac1_share', 'cac2_share', 'cac3_share', 'hf1_share',
       'hf2_share', 'hf3_share', 'op7_share', 'pn3_share', 'pn6_share',
       'scipcard2_share', 'scipinf10_share', 'scipinf1_share',
       'scipinf2_share', 'scipinf3_share', 'scipinf4_share', 'sc

The meanings of these column names can be found in the crosswalk:

In [45]:
poc_xw = pd.read_csv('../hospital-compare-master/misc/poc_xwlk.csv')
poc_xw.head()

,measurecode,condition,measurename
0,ami1,Heart Attack,Patients Given Aspirin at Arrival
1,ami2,Heart Attack,Patients Given Aspirin at Discharge
2,ami3,Heart Attack,Patients Given ACE Inhibitor for Left Ventricu...
3,ami3,Heart Attack,Patients Given ACE Inhibitor or ARB for Left V...
4,ami4,Heart Attack,Patients Given Adult Smoking Cessation Advice/...


HCAHPS data:

In [46]:
hosp_quality_hcahps = pd.read_csv('../hospital-compare-master/output/hcahps.csv').rename(columns={'pn':'CCN','year':'Year'})

In [47]:
hosp_quality_hcahps.columns

Index(['CCN', 'nsurveys', 'rrate', 'clean_score', 'commdoc_score',
       'commnurse_score', 'explain_score', 'help_score', 'info_score',
       'overall_score', 'pain_score', 'quiet_score', 'recommend_score', 'Year',
       'understood_score'],
      dtype='object')

The meanings of these column names can be found in the crosswalk:

In [48]:
hcahps_xw = pd.read_csv('../hospital-compare-master/misc/hcahps_xwlk.csv')
hcahps_xw.head()

,questionno,question,answerdescription,measurecode,meas,value,note
0,1,How do patients rate the hospital overall?,Patients who gave a rating of 6 or lower (low),H_HSP_RATING_0_6,overall,1,NaN
1,1,How do patients rate the hospital overall?,Patients who gave a rating of 7 or 8 (medium),H_HSP_RATING_7_8,overall,2,NaN
2,1,How do patients rate the hospital overall?,Patients who gave a rating of 9 or 10 (high),H_HSP_RATING_9_10,overall,3,NaN
3,2,How often did doctors communicate well with pa...,Doctors always communicated well,H_COMP_2_A_P,commdoc,3,NaN
4,2,How often did doctors communicate well with pa...,Doctors sometimes or never communicated well,H_COMP_2_SN_P,commdoc,1,NaN


In [49]:
hcahps_xw_cb = pd.read_csv('../hospital-compare-master/misc/hcahps_xwlk_codebook.csv')
hcahps_xw_cb.head()

,meas,question,answerdescription,value,note
0,overall,How do patients rate the hospital overall?,Patients who gave a rating of 9 or 10 (high),3,NaN
1,overall,How do patients rate the hospital overall?,Patients who gave a rating of 7 or 8 (medium),2,NaN
2,overall,How do patients rate the hospital overall?,Patients who gave a rating of 6 or lower (low),1,NaN
3,commdoc,How often did doctors communicate well with pa...,Doctors always communicated well,3,NaN
4,commdoc,How often did doctors communicate well with pa...,Doctors usually communicated well,2,NaN


Mortality and Readmission data:

In [50]:
hosp_quality_mort = pd.read_csv('../hospital-compare-master/output/mortreadm.csv').rename(columns={'pn':'CCN','year':'Year'})

In [51]:
hosp_quality_mort.columns

Index(['CCN', 'ami_mort_rate', 'ami_readm_rate', 'hf_mort_rate',
       'hf_readm_rate', 'pn_mort_rate', 'pn_readm_rate', 'ami_mort_npatients',
       'ami_readm_npatients', 'hf_mort_npatients', 'hf_readm_npatients',
       'pn_mort_npatients', 'pn_readm_npatients', 'Year', 'all_readm_rate',
       'hk_readm_rate', 'all_readm_npatients', 'hk_readm_npatients',
       'copd_mort_rate', 'copd_readm_rate', 'stk_mort_rate', 'stk_readm_rate',
       'copd_mort_npatients', 'copd_readm_npatients', 'stk_mort_npatients',
       'stk_readm_npatients', 'cabg_mort_rate', 'cabg_readm_rate',
       'cabg_mort_npatients', 'cabg_readm_npatients'],
      dtype='object')

Merging all hospital quality data together, then merging into the main dataset:

In [52]:
hosp_quality_poc['CCN'] = hosp_quality_poc['CCN'].astype(object)
hosp_quality_hcahps['CCN'] = hosp_quality_hcahps['CCN'].astype(object)
hosp_quality_mort['CCN'] = hosp_quality_mort['CCN'].astype(object)

In [53]:
intermediate = hosp_quality_poc.merge(hosp_quality_hcahps, how='outer', on=['CCN','Year'])

In [54]:
hosp_quality_full = intermediate.merge(hosp_quality_mort, how='outer', on=['CCN','Year'])

In [55]:
hospitals_full = hospitals_full.merge(hosp_quality_full, how='left', on=['CCN','Year'])

In [56]:
hospitals_full.shape

(34609, 181)

In [57]:
len(hospitals_full['CCN'].unique())

2235

### Gathering health resource/demographic data:

I'll get health resource and demographic data from the Area Health Resources Files (publicly available in the HRSA/Health Resources and Services Administration data warehouse: https://data.hrsa.gov/data/download). 

This dataset provides current as well as historic data for more than 6,000 variables for each of the nation’s counties, as well as state and national data. It contains information on health facilities, health professions, measures of resource scarcity, health status, economic activity, health training programs, and socioeconomic and environmental characteristics. 

In [58]:
ARHF_Full = pd.read_csv('../data/ARHF_Full.csv')

In [59]:
ARHF_Full['County_FIPS'] = ARHF_Full['County_FIPS'].astype(int).astype(str).str.zfill(3)
ARHF_Full['State_FIPS'] = ARHF_Full['State_FIPS'].astype(int).astype(str).str.zfill(2)


In [60]:
ARHF_Full[['State_FIPS','County_FIPS','State','County']]

,State_FIPS,County_FIPS,State,County
0,02,020,AK,ANCHORAGE (B)
1,02,020,AK,ANCHORAGE (B)
2,02,020,AK,ANCHORAGE (B)
3,02,020,AK,ANCHORAGE (B)
4,02,020,AK,ANCHORAGE (B)
...,...,...,...,...
9511,56,025,WY,NATRONA
9512,56,025,WY,NATRONA
9513,56,025,WY,NATRONA
9514,56,025,WY,NATRONA


In [61]:
ARHF_Full[['State_FIPS','County_FIPS','State','County']].dtypes

State_FIPS     object
County_FIPS    object
State          object
County         object
dtype: object

In [62]:
# Standardizing merge columns
hospitals_full['State_FIPS'] = hospitals_full['State_FIPS'].astype(int).astype(str).str.zfill(2)

In [63]:
hospitals_full['County_FIPS'].dtype

dtype('int64')

In [64]:
hospitals_full['County_FIPS'] = hospitals_full['County_FIPS'].astype(str).str.zfill(5)
hospitals_full['County_FIPS'] = hospitals_full['County_FIPS'].str[2:]


In [65]:
hospitals_full[['State_FIPS','County_FIPS','State','County']]

,State_FIPS,County_FIPS,State,County
0,22,113,LA,VERMILION
1,22,113,LA,VERMILION
2,22,113,LA,VERMILION
3,22,113,LA,VERMILION
4,22,113,LA,VERMILION
...,...,...,...,...
34604,06,075,CA,SANFRANCISCO
34605,06,075,CA,SANFRANCISCO
34606,06,075,CA,SANFRANCISCO
34607,06,075,CA,SANFRANCISCO


In [66]:
hospitals_full[['State_FIPS','County_FIPS','State','County']].dtypes

State_FIPS     object
County_FIPS    object
State          object
County         object
dtype: object

In [67]:
hospitals_full = hospitals_full.merge(ARHF_Full, how='left', on=['State_FIPS','County_FIPS','State','County','Year'])

In [68]:
hospitals_full.head()

,Year,CCN,"Donations, Land Improvements",NUMBER OF BEDS: Total Hospital,BALANCE SHEET: Total Current Assets (G_C1THRU4_11),Total Bad Debt expense,"RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)",BALANCE SHEET: Inventory (G_C1THRU4_7),BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),Net Revenue from Medicaid,...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"
0,2010,190034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000310,0.054363,0.001638,0.0,0.000707,0.170210,0.000034,17.5,57999.0,7.2
1,2011,190034,NaN,44.0,12705192.0,5211785.0,1618104.0,733401.0,935269.0,9286631.0,...,0.000309,0.044856,0.001630,0.0,0.000721,0.135510,0.000034,19.2,58276.0,6.6
2,2012,190034,NaN,44.0,14373750.0,5321114.0,1712816.0,847101.0,85345.0,6387241.0,...,0.000341,0.040342,0.001618,0.0,0.000834,0.110859,0.000034,17.4,58723.0,5.5
3,2013,190034,NaN,44.0,13509323.0,6059557.0,1759981.0,823868.0,598983.0,6895212.0,...,0.000338,0.045078,0.001603,0.0,0.000726,0.134204,0.000034,18.6,59253.0,NaN
4,2014,190034,NaN,44.0,13323600.0,6552984.0,1924140.0,857216.0,1060615.0,7477299.0,...,0.000335,0.046447,0.001594,0.0,0.000671,0.124178,0.000034,18.3,59616.0,5.6


In [69]:
hospitals_full.shape

(34609, 202)

In [70]:
print(hospitals_full.columns.tolist())

['Year', 'CCN', 'Donations, Land Improvements', 'NUMBER OF BEDS: Total Hospital', 'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)', 'Total Bad Debt expense', 'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)', 'BALANCE SHEET: Inventory (G_C1THRU4_7)', 'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)', 'Net Revenue from Medicaid', 'Total Charges', 'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 'IPPS Interim payment', 'Medicaid charges', 'Total Costs', 'Total Inpatient Days', 'ADJUSTED SALARIES, Subtotal Salaries', 'REIMBURSEMENT SETTLEMENT: Interim payments', 'Cost of Uncompensated Care', 'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)', 'BED DAYS: Total Hospital', 'BALANCE SHEET: Total Assets (G_C1THRU4_36)', 'NUMBER OF BEDS: ICU', 'NUMBER OF BEDS: Adults & Pediatrics', 'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)', 'Total cost of charity care', 'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)', 'IPPS Payment amount (un

In [71]:
hospitals_full.isna().sum()

Year                                                                         0
CCN                                                                          0
Donations, Land Improvements                                             34547
NUMBER OF BEDS: Total Hospital                                           23763
BALANCE SHEET: Total Current Assets (G_C1THRU4_11)                       23836
                                                                         ...  
Per Capita Total Medicare Inpatient Days Short Term General Hospitals    14161
Per Capita Total Number Hospitals                                        14161
Percent Persons in Poverty                                               12541
Population Estimate                                                      10497
Unemployment Rate, 16+                                                   12428
Length: 202, dtype: int64

In [72]:
hospitals_full['County_FIPS'].isna().sum()

np.int64(0)

In [73]:
# Confirming one hospital per year
hospitals_full[hospitals_full[['CCN','Year']].duplicated(keep=False)]

,Year,CCN,"Donations, Land Improvements",NUMBER OF BEDS: Total Hospital,BALANCE SHEET: Total Current Assets (G_C1THRU4_11),Total Bad Debt expense,"RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)",BALANCE SHEET: Inventory (G_C1THRU4_7),BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),Net Revenue from Medicaid,...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"


In [74]:
len(hospitals_full['CCN'].unique())

2235

Adjusting financial data for inflation (converting financial values into 2025 purchasing power using CPI index):

In [75]:
cols_to_update = ['BALANCE SHEET: Total Assets (G_C1THRU4_36)',
                  'REIMBURSEMENT SETTLEMENT: Interim payments', 
                  'HAC reduction adjustment amount', 
                  'IPPS Interim payment', 
                  'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)', 
                  'BALANCE SHEET: Total Current Liabilities (G_C1THRU4_45)', 
                  'Total cost of charity care', 
                  'HVBP payment adjustment amount', 
                  'Total Salaries', 
                  'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 
                  'Net Revenue from Medicaid', 
                  'Total Bad Debt expense', 
                  'Total Charges', 
                  'ADJUSTED SALARIES, Subtotal Salaries', 
                  'Donations, Land Improvements', 
                  'Total Costs', 
                  'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)', 
                  'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)', 
                  'REIMBURSEMENT SETTLEMENT: Subtotal', 
                   'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)', 
                    'STATEMENT OF REVENUES AND EXPENSES: Net Patient Revenue (G3_C1_3)', 
                  'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)', 
                  'BALANCE SHEET: Inventory (G_C1THRU4_7)', 
                  'IPPS Payment amount (unadjusted)', 
                  'BALANCE SHEET: Temporary Investments (G_C1THRU4_2)', 
                  'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)', 
                  'Cost of Uncompensated Care', 
                  'Medicaid charges', 
                  'Total Liabilities',
                  'Financial Indicators: Total Net Assets', 
                  'S-10 DATA: Medicaid Costs', 
                  'Financial Indicators: Cash Reserves', 
                  'Financial Indicators: Net Profit Margin', 
                  'Financial Indicators: Operating Profit', 
                  'Financial Indicators: Operating Profit Margin', 
                  'Financial Indicators: SOLVENCY total assets less total liabilities', 
                  'Median Household Income',
                  'Per Capita Personal Income'
                 ]

In [76]:
cpi = pd.read_excel('../data/CPI_Data.xlsx',skiprows=11)[['Year','Annual']]

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [77]:
cpi

,Year,Annual
0,2010,218.056
1,2011,224.939
2,2012,229.594
3,2013,232.957
4,2014,236.736
5,2015,237.017
6,2016,240.007
7,2017,245.120
8,2018,251.107
9,2019,255.657


In [78]:
cpi.dtypes

Year        int64
Annual    float64
dtype: object

In [79]:
cpi[cpi['Year']==2025]['Annual']

15    321.943
Name: Annual, dtype: float64

In [80]:
cpi['Scale Factor'] = cpi[cpi['Year']==2025]['Annual'].item() / cpi['Annual']

In [81]:
cpi

,Year,Annual,Scale Factor
0,2010,218.056,1.476423
1,2011,224.939,1.431246
2,2012,229.594,1.402227
3,2013,232.957,1.381985
4,2014,236.736,1.359924
5,2015,237.017,1.358312
6,2016,240.007,1.341390
7,2017,245.120,1.313410
8,2018,251.107,1.282095
9,2019,255.657,1.259277


In [82]:
cpi_dict = dict(zip(cpi["Year"], cpi["Scale Factor"]))

In [83]:
# Map the year column to the CPI scale factor using the dictionary
cpi_series = hospitals_full['Year'].map(cpi_dict)

# Multiply the target columns by the CPI series
for col in cols_to_update:
    hospitals_full[col] = hospitals_full[col] * cpi_series

Creating time target for survival regression. This represents the final year we have data for the hospital (either the year the hospital closed or the last year we have data on the hospital) minus the first year that we have data for the hospital

In [84]:
# Get the min/max year for each CCN
time_df = hospitals_full.groupby('CCN')['Year'].agg(['min', 'max']).reset_index()

# Calculate the difference
time_df['Time'] = time_df['max'] - time_df['min']

# Merge back to the original dataset
hospitals_full = hospitals_full.merge(time_df[['CCN', 'Time']], on='CCN', how='left')


In [85]:
# Save full dataset
hospitals_full.to_csv('../data/hospitals_full.csv', index=False)